In [1]:
import torch
import torch.nn as nn
import math

from sympy.physics.units import mol

In [2]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

In [3]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=2,
                 dim_ff=256, num_classes=2, max_len=512, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoder(x)
        x = self.encoder(x)
        x = x[:, 0, :]
        return self.classifier(x)

In [4]:
vocab_size = 1000

In [5]:
model = TransformerClassifier(vocab_size=vocab_size, num_classes=3)

In [6]:
print(model)

TransformerClassifier(
  (embedding): Embedding(1000, 128)
  (pos_encoder): PositionalEncoding()
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (classifier): Linear(in_features=128, out_features=3, bias=True)
)


In [7]:
x = torch.randint(0, vocab_size, (4, 20))

In [8]:
x

tensor([[327, 419, 515, 368, 987, 835, 573, 859, 363, 318, 441, 247, 504,  78,
          89, 848, 621, 646, 750,  32],
        [174,  56, 665, 578, 364, 808, 444, 521, 160,  81, 932, 274, 802, 687,
         921, 279, 534,  30, 752, 952],
        [163, 526, 518, 728, 876, 458, 164, 432, 521, 148, 520,  83, 329, 868,
         905, 494, 651, 574, 845, 816],
        [422, 123, 948, 644, 411, 226, 834, 305, 448, 944, 940, 188, 174, 824,
         924, 315, 570, 260, 360, 273]])

In [9]:
logits = model(x)

In [10]:
logits

tensor([[ 0.9104, -1.2304, -0.9136],
        [ 0.5448, -0.9611, -0.6119],
        [ 0.5647, -0.1940, -0.7854],
        [-0.7623,  0.0444, -0.6148]], grad_fn=<AddmmBackward0>)

In [11]:
print(f"输出形状: {logits.shape}")

输出形状: torch.Size([4, 3])


In [12]:
print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")

模型参数量: 393,347
